# Walk-Forward Validation

Reference: [/wiki/walk-forward-validation](/wiki/walk-forward-validation)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')


## Why k-fold leaks the future

k-fold randomly shuffles data, so a training sample at t=80 can appear
alongside a validation sample at t=60 — the model has seen the future.
Walk-forward validation always trains on the past and validates on the future.


## Worked trace: 12-observation series

In [ ]:
y = np.array([10,12,14,13,15,17,16,18,20,19,21,23], dtype=float)

# Naive predictor: mean of last 3 observations
def predict_mean3(y_train):
    return float(y_train[-3:].mean())

n0 = 6
errors = []
print(f'Round  Train     Pred  Actual  |Error|')
print('-' * 42)
for i, t in enumerate(range(n0, len(y))):
    pred = predict_mean3(y[:t])
    actual = y[t]
    err = abs(actual - pred)
    errors.append(err)
    print(f'{i+1:<7} 1-{t:<5} {pred:>6.2f}  {actual:>6.1f}  {err:>7.2f}')
print(f'\nMAE = {np.mean(errors):.2f}  (expected 2.00)')


## Expanding window vs sliding window

In [ ]:
def walk_forward_cv(y, model_fn, n0=None, h=1, window=None):
    T = len(y)
    if n0 is None: n0 = T // 2
    errors = []
    for t in range(n0, T - h + 1):
        start = max(0, t - window) if window else 0
        y_hat = model_fn(y[start:t])
        errors.append(abs(y[t] - y_hat))
    return np.mean(errors), len(errors)

# AR(1) series for comparison
rng = np.random.default_rng(0)
ar1 = np.zeros(100)
for t in range(1, 100):
    ar1[t] = 0.7 * ar1[t-1] + rng.normal()

naive_ar = lambda y_train: float(y_train[-1]) * 0.7
mae_exp, n_exp = walk_forward_cv(ar1, naive_ar, n0=30)
mae_slide, n_slide = walk_forward_cv(ar1, naive_ar, n0=30, window=20)
print(f'Expanding window  MAE={mae_exp:.3f}  ({n_exp} folds)')
print(f'Sliding  W=20     MAE={mae_slide:.3f}  ({n_slide} folds)')


## Visualising expanding windows

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
n0_vis, n_rounds = 20, 8
for i, t in enumerate(range(n0_vis, n0_vis + n_rounds)):
    ax.barh(i, t, left=0, height=0.6, color='steelblue', alpha=0.4 + i*0.05)
    ax.plot(t, i, 'o', color='orange', ms=8)
ax.set_xlabel('Time index'); ax.set_ylabel('CV round')
ax.set_title('Walk-forward expanding window: training set grows by 1 each round')
ax.plot([], [], 'o', color='orange', label='Validation point')
ax.barh([], [], color='steelblue', label='Training window')
ax.legend(); plt.tight_layout(); plt.show()


## ✏️ Your turn

Implement sliding-window CV with W=10 on the AR(1) series and report the MAE.


In [ ]:
# TODO(you): call walk_forward_cv with window=10
mae_w10 = None  # replace

assert mae_w10 is not None, 'compute mae_w10!'
print(f'Sliding W=10 MAE: {mae_w10:.3f}')


<details>
<summary>Solution</summary>

```python
mae_w10, _ = walk_forward_cv(ar1, naive_ar, n0=30, window=10)
```

</details>
